In [1]:
baseline_dir = "../../results/gpt_4.1-qwen25-32b-prod/"
api_adapter_dir = "../../results/gpt_4.1-surgical_adapter_v27_qwen3_8b_step_0"

In [2]:
import os
import json

In [3]:
# load all the json files
baseline_files = [f for f in os.listdir(baseline_dir) if f.endswith('.json')]
api_adapter_files = [f for f in os.listdir(api_adapter_dir) if f.endswith('.json')]

# json file is a list of dicts
baseline_data = [json.load(open(os.path.join(baseline_dir, f))) for f in baseline_files]
api_adapter_data = [json.load(open(os.path.join(api_adapter_dir, f))) for f in api_adapter_files]

In [4]:
# Analysis 1:
# Find out tasks which were solved in all of the runs in baseline
# FInd out tasks which failed in at least one of the runs in api_adapter
# Find the intersection of the two sets

from collections import defaultdict

solved_tasks = defaultdict(int)
for run in baseline_data:
    for task in run:
        if task['reward'] == 1:
            solved_tasks[task['task_id']] += 1

solved_tasks_in_all_baseline_runs_set = set([task_id for task_id, count in solved_tasks.items() if count == len(baseline_data)])

unsolved_tasks_in_at_least_one_api_adapter_run_set = set()
for run in api_adapter_data:
    for task in run:
        if task['reward'] == 0:
            unsolved_tasks_in_at_least_one_api_adapter_run_set.add(task['task_id'])

analysis_1_tasks = solved_tasks_in_all_baseline_runs_set & unsolved_tasks_in_at_least_one_api_adapter_run_set
print(f'Number of tasks solved in all baseline runs: {len(solved_tasks_in_all_baseline_runs_set)}')
print(f'Number of tasks unsolved in at least one api_adapter run: {len(unsolved_tasks_in_at_least_one_api_adapter_run_set)}')
print(f'Number of tasks solved in all baseline runs and unsolved in at least one api_adapter run: {len(analysis_1_tasks)}')
print('Tasks: ', analysis_1_tasks)

Number of tasks solved in all baseline runs: 30
Number of tasks unsolved in at least one api_adapter run: 101
Number of tasks solved in all baseline runs and unsolved in at least one api_adapter run: 19
Tasks:  {4, 12, 16, 19, 26, 35, 43, 47, 48, 53, 54, 60, 70, 73, 75, 87, 89, 91, 92}


In [5]:
# Analysis 2:
# Find out tasks which were solved at least once in baseline
# Find out tasks which were unsolved in all of the runs in api_adapter
# Find the intersection of the two set

solved_tasks_in_atleast_one_baseline_run_set = set()
for run in baseline_data:
    for task in run:
        if task['reward'] == 1:
            solved_tasks_in_atleast_one_baseline_run_set.add(task['task_id'])

unsolved_tasks_in_all_api_adapter_runs_set = defaultdict(int)
for run in api_adapter_data:
    for task in run:
        if task['reward'] == 0:
            unsolved_tasks_in_all_api_adapter_runs_set[task['task_id']] += 1

unsolved_tasks_in_all_api_adapter_runs_set = set([task_id for task_id, count in unsolved_tasks_in_all_api_adapter_runs_set.items() if count == len(api_adapter_data)])

analysis_2_tasks = solved_tasks_in_atleast_one_baseline_run_set & unsolved_tasks_in_all_api_adapter_runs_set
print(f'Number of tasks solved at least once in baseline: {len(solved_tasks_in_atleast_one_baseline_run_set)}')
print(f'Number of tasks unsolved in all api_adapter runs: {len(unsolved_tasks_in_all_api_adapter_runs_set)}')
print(f'Number of tasks solved at least once in baseline and unsolved in all api_adapter runs: {len(analysis_2_tasks)}')
print('Tasks: ', analysis_2_tasks)

Number of tasks solved at least once in baseline: 99
Number of tasks unsolved in all api_adapter runs: 20
Number of tasks solved at least once in baseline and unsolved in all api_adapter runs: 9
Tasks:  {66, 101, 102, 9, 74, 46, 81, 58, 29}


In [6]:
analysis_3_tasks = solved_tasks_in_all_baseline_runs_set & unsolved_tasks_in_all_api_adapter_runs_set
print(f'Number of tasks solved in all baseline runs and unsolved in all api_adapter runs: {len(analysis_3_tasks)}')
print('Tasks: ', analysis_3_tasks)


Number of tasks solved in all baseline runs and unsolved in all api_adapter runs: 0
Tasks:  set()


In [7]:
import plotly.graph_objects as go
from collections import defaultdict

# Calculate sum of rewards per task for baseline
baseline_task_rewards = defaultdict(int)
for run in baseline_data:
    for task in run:
        baseline_task_rewards[task['task_id']] += task['reward']

# Calculate sum of rewards per task for api_adapter
api_adapter_task_rewards = defaultdict(int)
for run in api_adapter_data:
    for task in run:
        api_adapter_task_rewards[task['task_id']] += task['reward']

# Get all unique task IDs and sort them
all_task_ids = sorted(set(baseline_task_rewards.keys()) | set(api_adapter_task_rewards.keys()))

# Prepare data for plotting
baseline_rewards = [baseline_task_rewards[task_id] for task_id in all_task_ids]
api_adapter_rewards = [api_adapter_task_rewards[task_id] for task_id in all_task_ids]

# Create the plot
fig = go.Figure()

# Add baseline trace
fig.add_trace(go.Scatter(
    x=all_task_ids,
    y=baseline_rewards,
    mode='lines+markers',
    name='Baseline',
    line=dict(color='blue', width=2),
    marker=dict(symbol='circle', size=6)
))

# Add api_adapter trace
fig.add_trace(go.Scatter(
    x=all_task_ids,
    y=api_adapter_rewards,
    mode='lines+markers',
    name='API Adapter',
    line=dict(color='red', width=2),
    marker=dict(symbol='circle', size=6)
))

# Update layout
fig.update_layout(
    title='Per Task Performance: Baseline vs API Adapter',
    xaxis_title='Task ID',
    yaxis_title='Sum of Rewards (All Runs)',
    width=900,
    height=500,
    showlegend=True,
    template='plotly_white'
)

# Show the plot
fig.show()


In [8]:
# analysis 4:
# find the tasks which were unsolved in all of the runs in baseline
# find the tasks which were solved in at least one run in api_adapter
# find the intersection of the two sets

unsolved_tasks_in_all_baseline_runs_set = defaultdict(int)
for run in baseline_data:
    for task in run:
        if task['reward'] == 0:
            unsolved_tasks_in_all_baseline_runs_set[task['task_id']] += 1
unsolved_tasks_in_all_baseline_runs_set = set([task_id for task_id, count in unsolved_tasks_in_all_baseline_runs_set.items() if count == len(baseline_data)])

solved_tasks_in_at_least_one_api_adapter_run_set = set()
for run in api_adapter_data:
    for task in run:
     if task['reward'] == 1:
            solved_tasks_in_at_least_one_api_adapter_run_set.add(task['task_id'])

analysis_4_tasks = unsolved_tasks_in_all_baseline_runs_set & solved_tasks_in_at_least_one_api_adapter_run_set
print(f'Number of tasks unsolved in all baseline runs: {len(unsolved_tasks_in_all_baseline_runs_set)}')
print(f'Number of tasks solved in at least one run in api_adapter: {len(solved_tasks_in_at_least_one_api_adapter_run_set)}')
print(f'Number of tasks unsolved in all baseline runs and solved in at least one run in api_adapter: {len(analysis_4_tasks)}')
print('Tasks: ', analysis_4_tasks)


Number of tasks unsolved in all baseline runs: 16
Number of tasks solved in at least one run in api_adapter: 95
Number of tasks unsolved in all baseline runs and solved in at least one run in api_adapter: 5
Tasks:  {98, 110, 17, 27, 31}
